# 01 · Tokenização: etiquetar o corpus com um modelo

Único notebook que **usa modelos de PLN**. Para o `MODELO` e o `CONTEXTO` escolhidos, ele:

1. prepara os poemas (separa os versos, normaliza maiúsculas, define os **tokens comuns**);
2. entrega o texto ao modelo conforme o contexto (`token`: cada palavra isolada; `verso`: cada verso; `poema`: o poema inteiro);
3. alinha as palavras do modelo aos tokens comuns pela posição dos caracteres;
4. grava, em `dados/analise/`:
   - `tokens_{contexto}_{modelo}.csv`: uma linha por token (`poema_id, verso_idx, token_idx, forma, lema, upos, morph, modelo, contexto`);
   - `lemas_{contexto}_{modelo}.csv`: correspondência forma → lema, para auditoria (só para modelos que lematizam);
   - uma linha em `registro.csv` (modelo, versão, acesso, contexto, normalização, data, contagens, divergências de alinhamento).

Os notebooks 02 (lexical) e 03 (morfológica) leem esses CSVs e **não rodam modelo nenhum**.

| Parâmetro | Valores |
|---|---|
| `MODELO` | `spacy`, `stanza`, `udpipe2`, `trankit`, `bertimbau_ft` (fase 2) |
| `CONTEXTO` | `token`, `verso`, `poema` |
| `NORMALIZAR_MAIUSCULAS` | `True` (padrão) / `False` |

**Glossário.** *forma*: a palavra como aparece no texto, em minúsculas. *lema*: forma de dicionário ("águas" → "água"). *UPOS*: classe gramatical do padrão Universal Dependencies. *morph*: traços `Feature=Valor`, ex.: `Gender=Masc|Number=Sing`.

## 0. Parâmetros

In [ ]:
# ===================== PARÂMETROS (edite aqui) =====================
MODELO = "spacy"                  # spacy | stanza | udpipe2 | trankit | bertimbau_ft (fase 2)
CONTEXTO = "poema"                # token | verso | poema
NORMALIZAR_MAIUSCULAS = True      # rebaixa a maiúscula inicial dos versos que continuam a frase anterior
ARQUIVO_CSV = "dados/bruto/poemas.csv"   # relativo à raiz do projeto 
RODAR_TESTE_NORMALIZACAO = False  # True: roda o teste do critério 6 

# Configuração de cada modelo (só a do MODELO escolhido é usada)
CONFIG_MODELOS = {
    "spacy":   {"pacote": "pt_core_news_lg"},                 # alternativas: pt_core_news_md, pt_core_news_sm
    "stanza":  {"idioma": "pt", "pacote": "default", "usar_gpu": False, "baixar": False},
    "udpipe2": {"acesso": "rest", "modelo": "portuguese-bosque-ud-2.12-230717", "lote": "bloco"},
    "trankit": {"idioma": "portuguese", "usar_gpu": False},
    "bertimbau_ft": {},                                       # fase 2
}
# ===================================================================

In [37]:
import sys
from pathlib import Path

# raiz do projeto: a pasta que contém src/ (funciona rodando de projeto/ ou de projeto/notebooks/)
RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
from IPython.display import display

from src import carga, preprocessamento, tokenizacao
from src.modelos import CAPACIDADES, criar_modelo

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

carga.validar_parametros(MODELO, CONTEXTO)
print(f"Modelo: {MODELO} | contexto: {CONTEXTO} | normalizar maiúsculas: {NORMALIZAR_MAIUSCULAS}")

Modelo: spacy | contexto: poema | normalizar maiúsculas: True


## 1. Corpus

Lê o CSV (`estrofes` com os versos separados por `/`, e `id`). Os metadados são ignorados; valores vazios viram texto vazio e espaços repetidos são colapsados.

In [38]:
corpus = carga.carregar_corpus(carga.RAIZ / ARQUIVO_CSV)
print(f"{len(corpus)} poemas; {int((corpus['texto'] == '').sum())} vazios")
display(corpus.head())

111 poemas; 0 vazios


,poema_id,texto
0,poema_001,"Fizeram uma parada, uma parada / Para o trem carregar café, / Antes, estradas difíceis..."
1,poema_002,"Antes da luz e da água encanada, / No povoado havia lamparina / E cisterna; dez a quin..."
2,poema_003,Saí das águas do mar / E nasci do cafezal de / Terra roxa. Passei a infância / No meu ...
3,poema_004,Nasci e montei na garupa / De muitos cavaleiros. Depois / Montei sozinho em cavalo de ...
4,poema_005,Num pé de café nasci. / O trenzinho passava / Por entre a plantação. Deu a hora / Exat...


### Como o texto chega ao modelo

- O `/` é removido; os versos são unidos por **um espaço** (nunca por quebra de linha).
- Com `NORMALIZAR_MAIUSCULAS = True`, a maiúscula inicial de um verso é rebaixada **quando o verso continua a frase do anterior** (o anterior não termina em `.`, `!`, `?` ou `…`), exceto para nomes próprios (palavras que aparecem com maiúscula no meio de algum verso do corpus). Maiúsculas no meio do verso nunca mudam.
- A normalização vale para os três contextos: no `verso` e no `token`, o verso ou a palavra já chega normalizado.

In [39]:
poemas = preprocessamento.preparar_corpus(corpus, NORMALIZAR_MAIUSCULAS)
protegidos = preprocessamento.capitalizadas_no_meio(corpus["texto"])
print(f"{len(protegidos)} palavras protegidas (maiúscula no meio do verso): {sorted(protegidos)[:15]}")

exemplo = next((p for p in poemas if p.versos), None)
if exemplo is not None:
    original = preprocessamento.separar_versos(corpus.loc[corpus["poema_id"] == exemplo.poema_id, "texto"].iloc[0])
    alterados = [(i, a, b) for i, (a, b) in enumerate(zip(original, exemplo.versos)) if a != b]
    print(f"\nPoema {exemplo.poema_id}: {len(original)} versos, {len(alterados)} com a maiúscula inicial rebaixada")
    display(pd.DataFrame(alterados, columns=["verso_idx", "original", "entregue ao modelo"]).head(10))
    print("Texto entregue ao modelo no contexto 'poema':\n", exemplo.texto[:400])

63 palavras protegidas (maiúscula no meio do verso): ['Abraão', 'Adão', 'Antônio', 'Belém', 'Bercheba', 'Canaã', 'Caná', 'Carlitiano', 'Cima', 'DE', 'DIA', 'Deus', 'Dickinson', 'Dona', 'Dragão']

Poema poema_001: 23 versos, 15 com a maiúscula inicial rebaixada


,verso_idx,original,entregue ao modelo
0,1,"Para o trem carregar café,","para o trem carregar café,"
1,2,"Antes, estradas difíceis, só carros de bois","antes, estradas difíceis, só carros de bois"
2,3,"Transitavam, levando dias e dias.","transitavam, levando dias e dias."
3,6,"Há rio, nem pedras.","há rio, nem pedras."
4,9,"Arame farpado, os divisores.","arame farpado, os divisores."
5,11,Cafeeira. Imenso céu azul circula,cafeeira. Imenso céu azul circula
6,12,O areal. Milhares de brancas nuvens,o areal. Milhares de brancas nuvens
7,13,Viajam. Caravanas luminosas,viajam. Caravanas luminosas
8,14,"Em movimento. O mais solitário, ali","em movimento. O mais solitário, ali"
9,15,Deixaria de sê-lo. Toda essa fagueira,deixaria de sê-lo. Toda essa fagueira


Texto entregue ao modelo no contexto 'poema':
 Fizeram uma parada, uma parada para o trem carregar café, antes, estradas difíceis, só carros de bois transitavam, levando dias e dias. Depois, uma casa aqui, outra ali. Formaram o povoado. Não há rio, nem pedras. De tijolos caiados e telhas antigas São as construções; de taipas e arame farpado, os divisores. Lugar arenoso no meio da terra roxa cafeeira. Imenso céu azul circula o areal. Milhares d


## 2. Capacidades do modelo

Quando o modelo não fornece lema ou traços, a coluna fica vazia no CSV e os notebooks 02 e 03 avisam e pulam o que depende dela.

In [40]:
display(pd.DataFrame(CAPACIDADES).T.rename_axis("modelo"))
print("Este modelo fornece:", CAPACIDADES[MODELO])

,upos,morph,lema
modelo,,,
spacy,True,True,True
stanza,True,True,True
udpipe2,True,True,True
trankit,True,True,True
bertimbau_ft,True,opcional,False


Este modelo fornece: {'upos': True, 'morph': True, 'lema': True}


## 3. Etiquetar o corpus

Cria o modelo (só aqui um modelo de PLN é carregado), etiqueta no contexto escolhido, alinha aos tokens comuns e grava os arquivos.

**Contrações** ("no", "pela", "dela") permanecem **um token**. Os modelos que as separam (Stanza, UDPipe 2, Trankit) são reagrupados: lema = componentes unidos por espaço (`em o`), classe = a da primeira componente. Proposta a confirmar (ver README).

In [41]:
modelo = criar_modelo(MODELO, **CONFIG_MODELOS[MODELO])
print("Versão:", modelo.versao(), "| acesso:", modelo.acesso())

resultado = tokenizacao.gerar_tokens(corpus, modelo, CONTEXTO, NORMALIZAR_MAIUSCULAS)
tokens = resultado.tokens
reg = resultado.registro
print(f"{reg['n_poemas']} poemas, {reg['n_tokens']} tokens, {reg['n_reagrupados']} contrações/palavras reagrupadas, "
      f"{reg['n_divergencias']} divergências de alinhamento")
print("Arquivo:", carga.caminho_tokens(CONTEXTO, MODELO))
display(tokens.head(12))

Versão: pt_core_news_lg 3.8.0; spacy 3.8.16 | acesso: local
111 poemas, 9683 tokens, 0 contrações/palavras reagrupadas, 5 divergências de alinhamento
Arquivo: C:\Users\Usuário\Desktop\Portinari_analise\dados\analise\tokens_poema_spacy.csv


,poema_id,verso_idx,token_idx,forma,lema,upos,morph,modelo,contexto
0,poema_001,0,0,fizeram,fizeram,VERB,Mood=Ind|Number=Plur|Person=3|Tense=Past|VerbForm=Fin,spacy,poema
1,poema_001,0,1,uma,um,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=Art,spacy,poema
2,poema_001,0,2,parada,parada,NOUN,Gender=Fem|Number=Sing,spacy,poema
3,poema_001,0,3,uma,um,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=Art,spacy,poema
4,poema_001,0,4,parada,parada,NOUN,Gender=Fem|Number=Sing,spacy,poema
5,poema_001,1,5,para,para,ADP,,spacy,poema
6,poema_001,1,6,o,o,DET,Definite=Def|Gender=Masc|Number=Sing|PronType=Art,spacy,poema
7,poema_001,1,7,trem,trem,NOUN,Gender=Masc|Number=Sing,spacy,poema
8,poema_001,1,8,carregar,carregar,VERB,VerbForm=Inf,spacy,poema
9,poema_001,1,9,café,café,NOUN,Gender=Masc|Number=Sing,spacy,poema


## 4. Verificações

**Divergências de alinhamento**: tokens em que os limites do modelo não coincidem com o token comum (ou que ficaram sem correspondência). Ficam em `divergencias_{contexto}_{modelo}.csv`.

In [42]:
if resultado.divergencias.empty:
    print("Nenhuma divergência de alinhamento.")
else:
    display(resultado.divergencias.head(20))

,poema_id,token_idx,token,motivo,palavras_modelo
0,poema_025,42,barba,limites não coincidem,barba-[251:257]
1,poema_041,20,barra,limites não coincidem,barra-[135:141]
2,poema_107,68,º,limites não coincidem,1º[376:378]
3,poema_108,88,º,limites não coincidem,1º[491:493]
4,poema_109,68,º,limites não coincidem,1º[376:378]


**Consistência entre arquivos**: todos os `tokens_*.csv` da pasta (outros modelos e contextos) devem ter exatamente os mesmos tokens.

In [43]:
display(tokenizacao.conferir_consistencia())

,arquivo,n_tokens,mesmos_tokens,so_neste,faltando
0,tokens_poema_spacy.csv,9683,True,0,0
1,tokens_poema_stanza.csv,9683,True,0,0
2,tokens_poema_udpipe2.csv,9683,True,0,0
3,tokens_verso_spacy.csv,9683,True,0,0
4,tokens_verso_udpipe2.csv,9683,True,0,0


**Registro** das execuções (modelo, versão exata, acesso, data...).

In [44]:
display(pd.read_csv(carga.PASTA_ANALISE / "registro.csv").tail(6))

,data,modelo,versao,acesso,contexto,normalizar_maiusculas,n_poemas,n_tokens,n_reagrupados,n_divergencias,arquivo_tokens
8,2026-10-23T13:13:14,udpipe2,udpipe2 rest; modelo=portuguese-bosque-ud-2.12-230717,servico online (LINDAT),verso,True,111,9683,899,541,tokens_verso_udpipe2.csv
9,2026-10-23T13:13:37,udpipe2,udpipe2 rest; modelo=portuguese-bosque-ud-2.12-230717,servico online (LINDAT),verso,True,111,9683,899,541,tokens_verso_udpipe2.csv
10,2026-10-23T14:26:24,spacy,pt_core_news_lg 3.8.0; spacy 3.8.16,local,verso,True,111,9683,0,5,tokens_verso_spacy.csv
11,2026-10-23T14:32:28,spacy,pt_core_news_lg 3.8.0; spacy 3.8.16,local,verso,True,111,9683,0,5,tokens_verso_spacy.csv
12,2026-10-23T14:33:36,spacy,pt_core_news_lg 3.8.0; spacy 3.8.16,local,poema,True,111,9683,0,5,tokens_poema_spacy.csv
13,2026-10-23T14:34:25,spacy,pt_core_news_lg 3.8.0; spacy 3.8.16,local,poema,True,111,9683,0,5,tokens_poema_spacy.csv


**Auditoria dos lemas**: formas com mais de um lema (o lematizador depende do contexto e da classe) e verbos que ficaram com lema igual à forma (provavelmente não lematizados).

In [45]:
if resultado.lemas is None:
    print("Este modelo não lematiza: nada a auditar.")
else:
    lemas = resultado.lemas
    varios = lemas.groupby("forma")["lema"].agg(lambda s: sorted(set(s)))
    varios = varios[varios.map(len) > 1]
    print(f"{len(varios)} formas com mais de um lema:")
    display(varios.head(15).rename("lemas"))
    suspeitos = lemas[(lemas["upos"] == "VERB") & (lemas["forma"] == lemas["lema"])]
    print(f"\n{len(suspeitos)} verbos com lema igual à forma:")
    display(suspeitos.head(15))

141 formas com mais de um lema:


forma
a                                     [a, ela, o]
acesos                            [aceso, acesos]
acontecimentos    [acontecimento, acontecimentos]
acordei                        [acordar, acordei]
alegres                           [alegr, alegre]
alguma                            [algum, alguma]
almas                               [alma, almas]
amanheci                    [amanhecer, amanheci]
amei                                 [amar, amei]
andei                             [andeer, andei]
ao                                      [a o, ao]
apagado                         [apagado, apagar]
as                                      [elas, o]
assassinados            [assassinado, assassinar]
caminhemos                   [caminh, caminhemos]
Name: lemas, dtype: object


375 verbos com lema igual à forma:


,forma,upos,lema,n
73,ver,VERB,ver,16
192,saber,VERB,saber,7
271,morrer,VERB,morrer,5
281,pedirão,VERB,pedirão,5
293,vejo,VERB,vejo,5
297,vinham,VERB,vinham,5
328,existirá,VERB,existirá,4
335,fugir,VERB,fugir,4
368,sair,VERB,sair,4
387,vi,VERB,vi,4


## 5. Teste da normalização de maiúsculas (critério de aceitação 6)

Para cada contexto, roda com `NORMALIZAR_MAIUSCULAS` em `True` e em `False` e compara a proporção de PROPN e uma amostra das palavras cuja classe mudou. **Não grava arquivos.** O resultado deve ser documentado no README para fixar o padrão recomendado por contexto. Ative com `RODAR_TESTE_NORMALIZACAO = True`.

In [46]:
if RODAR_TESTE_NORMALIZACAO:
    resumo, amostra = tokenizacao.comparar_normalizacao(corpus, modelo)
    display(resumo)
    display(amostra)
else:
    print("Teste da normalização desligado (RODAR_TESTE_NORMALIZACAO = False).")

Teste da normalização desligado (RODAR_TESTE_NORMALIZACAO = False).
